In [1]:
# Cell 1 - Imports
import pandas as pd
import numpy as np
import requests
import time
from tmdbv3api import TMDb, Movie

In [2]:
# Cell 2 - Setup TMDB
tmdb = TMDb()
tmdb.api_key = '8f1244df6398593cab5fa022446db4cf'
tmdb_movie = Movie()

In [3]:
# Cell 3 - Helper Functions
def get_genre(x):
    for attempt in range(3):
        try:
            result = tmdb_movie.search(x)
            if not result:
                return np.nan
            movie_id = result[0].id
            response = requests.get(
                f"https://api.themoviedb.org/3/movie/{movie_id}?api_key={tmdb.api_key}",
                timeout=10
            )
            data_json = response.json()
            genres = [g['name'] for g in data_json.get('genres', [])]
            time.sleep(0.5)
            return " ".join(genres) if genres else np.nan
        except Exception as e:
            print(f"Attempt {attempt+1} failed for '{x}': {e}")
            time.sleep(2 ** attempt)
    return np.nan

def get_director(x):
    if " (director)" in x:
        return x.split(" (director)")[0]
    elif " (directors)" in x:
        return x.split(" (directors)")[0]
    else:
        return x.split(" (director/screenplay)")[0]

def get_actor1(x):
    return ((x.split("screenplay); ")[-1]).split(", ")[0])

def get_actor2(x):
    if len((x.split("screenplay); ")[-1]).split(", ")) < 2:
        return np.nan
    return ((x.split("screenplay); ")[-1]).split(", ")[1])

def get_actor3(x):
    if len((x.split("screenplay); ")[-1]).split(", ")) < 3:
        return np.nan
    return ((x.split("screenplay); ")[-1]).split(", ")[2])

In [4]:
# Cell 4 - Main Pipeline Function
def scrape_and_process_movies(year):
    print(f"\n========== Processing {year} ==========\n")
    link = f"https://en.wikipedia.org/wiki/List_of_American_films_of_{year}"
    df = pd.read_html(link, header=0, storage_options={"User-Agent": "Mozilla/5.0"})
    df = pd.concat([df[2], df[3], df[4], df[5]])

    genres_list = []
    for i, title in enumerate(df['Title']):
        print(f"{i+1}/{len(df)}: {title}")
        genres_list.append(get_genre(str(title)))

    df['genres'] = genres_list
    df = df[['Title', 'Cast and crew', 'genres']]

    df['director_name'] = df['Cast and crew'].map(lambda x: get_director(str(x)))
    df['actor_1_name']  = df['Cast and crew'].map(lambda x: get_actor1(str(x)))
    df['actor_2_name']  = df['Cast and crew'].map(lambda x: get_actor2(str(x)))
    df['actor_3_name']  = df['Cast and crew'].map(lambda x: get_actor3(str(x)))

    df = df.rename(columns={'Title': 'movie_title'})
    df = df[['director_name', 'actor_1_name', 'actor_2_name',
             'actor_3_name', 'genres', 'movie_title']]

    df['actor_2_name'] = df['actor_2_name'].replace(np.nan, 'unknown')
    df['actor_3_name'] = df['actor_3_name'].replace(np.nan, 'unknown')
    df['movie_title']  = df['movie_title'].str.lower()

    df['comb'] = (df['actor_1_name'] + ' ' + df['actor_2_name'] + ' ' +
                  df['actor_3_name'] + ' ' + df['director_name'] + ' ' + df['genres'])
    return df

In [8]:
# Cell 5 - Run only 2018, 2019, 2020
import os

all_dfs = []

for year in range(2018, 2021):  # 2021 is excluded, so only 2018, 2019, 2020
    checkpoint_file = f'checkpoint_{year}.csv'
    
    if os.path.exists(checkpoint_file):
        print(f"✅ {year} already done, loading from checkpoint...")
        df = pd.read_csv(checkpoint_file)
        all_dfs.append(df)
        continue
    
    df = scrape_and_process_movies(year)
    df.to_csv(checkpoint_file, index=False)
    print(f"💾 {year} checkpoint saved!")
    all_dfs.append(df)

# merge with old data
my_df = pd.concat(all_dfs, ignore_index=True)
old_df = pd.read_csv('new_data.csv')
final_df = pd.concat([old_df, my_df], ignore_index=True)
final_df = final_df.dropna(how='any')
final_df.to_csv('main_data.csv', index=False)
print(f"✅ Done! Total movies: {len(final_df)}")


========== Processing 2018 ==========

1/251: Insidious: The Last Key
2/251: The Strange Ones
3/251: The Commuter
4/251: Proud Mary
5/251: Acts of Violence
6/251: Freak Show
7/251: Humor Me
8/251: 12 Strong
9/251: Den of Thieves
10/251: Forever My Girl
11/251: Thane of East County
12/251: Maze Runner: The Death Cure
13/251: Please Stand By
14/251: Winchester
15/251: A Fantastic Woman
16/251: Armed
17/251: The Cloverfield Paradox
18/251: Bad Apples
19/251: Peter Rabbit
20/251: Pad Man
21/251: Fifty Shades Freed
22/251: The 15:17 to Paris
23/251: Permission
24/251: Golden Exits
25/251: Black Panther
26/251: Looking Glass
27/251: Nostalgia
28/251: Samson
29/251: Game Night
30/251: Annihilation
31/251: Every Day
32/251: The Cured
33/251: Red Sparrow
34/251: Death Wish
Attempt 1 failed for 'Death Wish': ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))
35/251: The Vanishing of Sidney Hall
36/251: Picking

In [9]:
final_df

,director_name,actor_1_name,actor_2_name,actor_3_name,genres,movie_title,comb
0,James Cameron,CCH Pounder,Joel David Moore,Wes Studi,Action Adventure Fantasy Sci-Fi,avatar,CCH Pounder Joel David Moore Wes Studi James C...
1,Gore Verbinski,Johnny Depp,Orlando Bloom,Jack Davenport,Action Adventure Fantasy,pirates of the caribbean: at world's end,Johnny Depp Orlando Bloom Jack Davenport Gore ...
2,Sam Mendes,Christoph Waltz,Rory Kinnear,Stephanie Sigman,Action Adventure Thriller,spectre,Christoph Waltz Rory Kinnear Stephanie Sigman ...
3,Christopher Nolan,Tom Hardy,Christian Bale,Joseph Gordon-Levitt,Action Thriller,the dark knight rises,Tom Hardy Christian Bale Joseph Gordon-Levitt ...
4,Doug Walker,Doug Walker,Rob Walker,unknown,Documentary,star wars: episode vii - the force awakens ...,Doug Walker Rob Walker unknown Doug Walker Doc...
...,...,...,...,...,...,...,...
6140,Robert Rodriguez,Priyanka Chopra Jonas,Pedro Pascal,YaYa Gosselin,Family Action Fantasy Comedy,we can be heroes,Priyanka Chopra Jonas Pedro Pascal YaYa Gossel...
6141,Paul Greengrass,Tom Hanks,Helena Zengel,unknown,Drama Western Adventure,news of the world,Tom Hanks Helena Zengel unknown Paul Greengras...
6142,Regina King,Kingsley Ben-Adir,Eli Goree,Aldis Hodge,Drama,one night in miami...,Kingsley Ben-Adir Eli Goree Aldis Hodge Regina...
6143,Emerald Fennell,Carey Mulligan,Bo Burnham,Alison Brie,Thriller Crime Drama,promising young woman,Carey Mulligan Bo Burnham Alison Brie Emerald ...
